In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
df = pd.read_csv("C:\\Users\\delta\\Predictive-maintenence-iot\\dataset\\ai4i2020.csv")
# Prepare features
target = "Machine failure"
X = df.drop(columns=["UDI", "Product ID", target])
y = df[target]
X = pd.get_dummies(X,columns=["Type"],drop_first=True)
# Split data
X_train, X_test, y_train, y_test = train_test_split( X,y,test_size=0.20,random_state=42,stratify=y)
# Create pipeline
pipeline = Pipeline([("smote", SMOTE(random_state=42)),("model", LogisticRegression(max_iter=1000))])
# Train model
pipeline.fit(X_train, y_train)

In [ ]:
failure_probability = pipeline.predict_proba(X_test)[:,1]
print("Prediction probabilities extracted successfully.")
print(failure_probability[:10])

In [ ]:
from sklearn.metrics import precision_recall_curve
precision, recall, thresholds = precision_recall_curve(y_test,failure_probability)
print("Precision-Recall values generated.")
print("Precision Length :", len(precision))
print("Recall Length    :", len(recall))
print("Threshold Length :", len(thresholds))

In [ ]:
import numpy as np
pr_table = pd.DataFrame({"Threshold": np.append(thresholds, np.nan),"Precision": precision,"Recall": recall})
pr_table.head(15)

In [ ]:
from sklearn.metrics import average_precision_score
average_precision = average_precision_score(y_test,failure_probability)
print("Average Precision Score")
print(round(average_precision,4))

In [ ]:
from matplotlib import pyplot as plt
plt.figure(figsize=(8,6))
plt.plot(recall,precision,linewidth=2,label=f"AP = {average_precision:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay
display = PrecisionRecallDisplay(precision=precision,recall=recall,average_precision=average_precision)
display.plot()
plt.title("Precision-Recall Curve")
plt.grid(True)
plt.show()

In [ ]:
threshold_table = pd.DataFrame({"Threshold":thresholds,"Precision":precision[:-1],"Recall":recall[:-1]})
threshold_table.head(20)

In [ ]:
high_precision = threshold_table[threshold_table["Precision"]>=0.80]
high_precision.head(15)

In [ ]:
high_recall = threshold_table[threshold_table["Recall"]>=0.80]
high_recall.head(15)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(thresholds,precision[:-1],color="blue",label="Precision")
plt.xlabel("Threshold")
plt.ylabel("Precision")
plt.title("Precision vs Threshold")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(thresholds,recall[:-1],color="green",label="Recall")
plt.xlabel("Threshold")
plt.ylabel("Recall")
plt.title("Recall vs Threshold")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(9,5))
plt.plot(thresholds,precision[:-1],label="Precision")
plt.plot(thresholds,recall[:-1],label="Recall")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Precision and Recall Trade-off")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
f1_scores = (2 *precision[:-1] *recall[:-1]) / (precision[:-1] +recall[:-1] +1e-10)
best_index = np.argmax(f1_scores)
print("Best Threshold")
print(round(thresholds[best_index],3))
print("Precision")
print(round(precision[best_index],3))
print("Recall")
print(round(recall[best_index],3))
print("F1 Score")
print(round(f1_scores[best_index],3))

In [ ]:
best_threshold = pd.DataFrame({"Metric":["Threshold","Precision","Recall","F1 Score"],
                               "Value":[thresholds[best_index],precision[best_index],recall[best_index],f1_scores[best_index]]})
best_threshold

In [ ]:
print("="*60)
print("PRECISION-RECALL INTERPRETATION")
print("="*60)
print(f"Average Precision Score : {average_precision:.4f}")
if average_precision > 0.80:
    print("Excellent Precision-Recall Performance")

elif average_precision > 0.60:
    print("Good Precision-Recall Performance")

else:
    print("Model Needs Improvement")

In [ ]:
pr_table.to_csv("precision_recall_results.csv",index=False)
print("Results exported successfully.")

In [ ]:
report = f"""
===========================================
Week 4 Day 2 Report
===========================================
Task Completed

Generated Precision-Recall Curve

Calculated Average Precision Score

Analyzed Threshold Performance

Compared Precision and Recall

Identified Best Threshold

Results

Average Precision Score :
{average_precision:.4f}

Best Threshold :
{thresholds[best_index]:.4f}

Best Precision :
{precision[best_index]:.4f}

Best Recall :
{recall[best_index]:.4f}

Best F1 Score :
{f1_scores[best_index]:.4f}

Conclusion
The Precision-Recall Curve provides a better evaluation than accuracy for this imbalanced predictive maintenance dataset. The selected threshold offers a balanced trade-off between detecting machine failures (Recall) and minimizing false alarms (Precision).
===========================================
"""
print(report)